# Investigacion de DGA y NXDOMAIN

## Objetivo

Identificar clientes con alto volumen de respuestas NXDOMAIN o dominios fallidos distintos.

## Entradas esperadas

- IP opcional.
- Ventana de tiempo.

## Requisitos

- Acceso al area de trabajo de Microsoft Sentinel.
- Funciones KQL publicadas: `fn_Normalize_Windows_DHCP`, `fn_Normalize_Windows_DNS`, `fn_Correlate_DHCP_DNS`.
- Paquetes Python sugeridos: `msticpy`, `pandas`, `matplotlib`, `plotly`, `networkx` segun el notebook.

## Secciones

1. NXDOMAIN por cliente.
2. Dominios fallidos distintos.
3. Tendencia por hora.
4. Candidatos a DGA.


In [ ]:
# Configuracion general - ajustar antes de ejecutar
workspace_id = "REEMPLAZAR_CON_WORKSPACE_ID"
tenant_id = "REEMPLAZAR_CON_TENANT_ID"

# Conexion sugerida con MSTICPy
# import msticpy as mp
# mp.init_notebook(namespace=globals())
# qry_prov = mp.QueryProvider("MSSentinel")
# qry_prov.connect(workspace=workspace_id, tenant_id=tenant_id)


In [ ]:
query_nxdomain = """
let Lookback = 7d;
let TargetIp = "REEMPLAZAR_CON_IP_OPCIONAL";
fn_Correlate_DHCP_DNS(Lookback)
| extend Response = toupper(tostring(ResponseCode))
| where Response has_any ("NXDOMAIN", "NAME_ERROR", "3") or RawMessage has_any ("NXDOMAIN", "Name Error")
| where TargetIp == "REEMPLAZAR_CON_IP_OPCIONAL" or ClientIp == TargetIp
| summarize NxdomainCount=count(), DistinctFailedDomains=dcount(QueryName), SampleDomains=make_set(QueryName, 30) by bin(TimeGenerated, 1h), ClientIp, HostName, ClientMac
| order by NxdomainCount desc
"""
# nxdomain_df = qry_prov.exec_query(query_nxdomain)
print(query_nxdomain)


## Resumen para incidente

Documentar aqui:

- Hallazgos principales.
- Entidades relevantes: IP, hostname, direccion MAC, dominio.
- Evidencia KQL usada.
- Recomendacion: cerrar, monitorear, escalar o contener.
